# 🚀 S6E4 - BALANCED ACCURACY BEAST (FAST)
### XGB + LGBM | Balanced Class Weights | Optimized for ~0.98+

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print('='*70)
print('🚀 S6E4 - BALANCED ACCURACY BEAST')
print('='*70)

In [ ]:
print('\n[1/5] Loading data...')
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
print(f'  Train: {train.shape}, Test: {test.shape}')
print(f'  Target: {train["Irrigation_Need"].value_counts().to_dict()}')

In [ ]:
print('\n[2/5] Feature engineering...')

train['is_train'] = True
test['is_train'] = False
test['Irrigation_Need'] = 'Low'
df = pd.concat([train, test], axis=0, ignore_index=True)

# Encode categoricals
cat_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Interaction features
df['Soil_pH_Soil_Moisture'] = df['Soil_pH'] * df['Soil_Moisture']
df['Temperature_Humidity'] = df['Temperature_C'] * df['Humidity']
df['Rainfall_Soil_Moisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
df['Sunlight_Temperature'] = df['Sunlight_Hours'] * df['Temperature_C']
df['Wind_Humidity'] = df['Wind_Speed_kmh'] * df['Humidity']
df['Organic_Temperature'] = df['Organic_Carbon'] * df['Temperature_C']
df['EC_Moisture'] = df['Electrical_Conductivity'] * df['Soil_Moisture']

# Ratio features
df['Organic_to_Soil'] = df['Organic_Carbon'] / (df['Soil_Moisture'] + 1e-6)
df['EC_to_Soil_pH'] = df['Electrical_Conductivity'] / (df['Soil_pH'] + 1e-6)
df['Rainfall_per_Hectare'] = df['Rainfall_mm'] / (df['Field_Area_hectare'] + 1e-6)
df['Prev_Irrigation_per_Hectare'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + 1e-6)
df['Temp_Humidity_Ratio'] = df['Temperature_C'] / (df['Humidity'] + 1e-6)
df['Rainfall_Humidity'] = df['Rainfall_mm'] / (df['Humidity'] + 1e-6)

# Polynomial features
df['Soil_pH_sq'] = df['Soil_pH'] ** 2
df['Moisture_sq'] = df['Soil_Moisture'] ** 2
df['Temperature_sq'] = df['Temperature_C'] ** 2
df['Humidity_sq'] = df['Humidity'] ** 2
df['Rainfall_log'] = np.log1p(df['Rainfall_mm'])
df['Prev_Irrigation_log'] = np.log1p(df['Previous_Irrigation_mm'])

# Domain features
df['Water_Stress'] = (df['Temperature_C'] * df['Wind_Speed_kmh']) / (df['Humidity'] + df['Soil_Moisture'] + 1e-6)
df['Irrigation_Efficiency'] = df['Previous_Irrigation_mm'] / (df['Rainfall_mm'] + 1e-6)
df['Crop_Water_Demand'] = df['Sunlight_Hours'] * df['Temperature_C'] / (df['Humidity'] + 1e-6)
df['Evapotranspiration_Proxy'] = (df['Temperature_C'] * df['Sunlight_Hours'] * df['Wind_Speed_kmh']) / (df['Humidity'] + 1e-6)
df['Soil_Water_Retention'] = df['Soil_Moisture'] * df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 1e-6)

# Binned features
df['Soil_Moisture_bin'] = pd.qcut(df['Soil_Moisture'], q=10, labels=False, duplicates='drop')
df['Rainfall_bin'] = pd.qcut(df['Rainfall_mm'], q=10, labels=False, duplicates='drop')
df['Temperature_bin'] = pd.qcut(df['Temperature_C'], q=10, labels=False, duplicates='drop')
df['Humidity_bin'] = pd.qcut(df['Humidity'], q=10, labels=False, duplicates='drop')

# Target encoding
for group_col in ['Crop_Type', 'Soil_Type', 'Region', 'Season', 'Irrigation_Type']:
    train_mask = df['is_train'] == True
    group_means = df[train_mask].groupby(group_col)['Irrigation_Need'].apply(
        lambda x: (x == 'High').mean() * 2 + (x == 'Medium').mean()
    )
    df[f'{group_col}_target_mean'] = df[group_col].map(group_means)
    group_counts = df[train_mask].groupby(group_col).size()
    df[f'{group_col}_count'] = df[group_col].map(group_counts)
    group_high = df[train_mask].groupby(group_col)['Irrigation_Need'].apply(
        lambda x: (x == 'High').mean()
    )
    df[f'{group_col}_high_prob'] = df[group_col].map(group_high)

# Cross features
df['Crop_Soil'] = df['Crop_Type'].astype(str) + '_' + df['Soil_Type'].astype(str)
df['Crop_Region'] = df['Crop_Type'].astype(str) + '_' + df['Region'].astype(str)
df['Crop_Season'] = df['Crop_Type'].astype(str) + '_' + df['Season'].astype(str)
df['Soil_Region'] = df['Soil_Type'].astype(str) + '_' + df['Region'].astype(str)
cross_cols = ['Crop_Soil', 'Crop_Region', 'Crop_Season', 'Soil_Region']
for col in cross_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# Split
train_final = df.iloc[:len(train)].copy()
test_final = df.iloc[len(train):].copy()
train_final['Irrigation_Need'] = train['Irrigation_Need']
train_final.drop(['is_train', 'id'], axis=1, inplace=True)
test_final.drop(['is_train'], axis=1, inplace=True)
if 'Irrigation_Need' in test_final.columns:
    test_final.drop(['Irrigation_Need'], axis=1, inplace=True)
if 'id' in test_final.columns:
    test_final.drop(['id'], axis=1, inplace=True)

print(f'  Features: {train_final.shape[1] - 1}')

In [ ]:
print('\n[3/5] Preparing...')
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(train_final['Irrigation_Need'])
X = train_final.drop('Irrigation_Need', axis=1)
X_test = test_final.copy()
print(f'  Classes: {target_encoder.classes_}')
print(f'  Features: {X.shape[1]}')

In [ ]:
print('\n[4/5] Training...')

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
test_xgb = np.zeros((len(X_test), 3))
test_lgb = np.zeros((len(X_test), 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\n  Fold {fold+1}/{N_SPLITS}')
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # XGBoost with balanced class weights
    xgb_model = xgb.XGBClassifier(
        n_estimators=1000, max_depth=7, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        gamma=0.2, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, eval_metric='mlogloss',
        tree_method='hist', verbosity=0
    )
    sample_weights = compute_sample_weight('balanced', y_train)
    xgb_model.fit(X_train, y_train, sample_weight=sample_weights,
                 eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = xgb_model.predict(X_val)
    bal_acc_xgb = balanced_accuracy_score(y_val, oof_xgb[val_idx])
    print(f'    XGB: {bal_acc_xgb:.6f}')
    
    # LightGBM with balanced class weights
    lgb_model = lgb.LGBMClassifier(
        n_estimators=1000, max_depth=7, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=15,
        reg_alpha=0.5, reg_lambda=2.0, class_weight='balanced',
        random_state=42, verbosity=-1
    )
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(100, verbose=False),
                            lgb.log_evaluation(0)])
    oof_lgb[val_idx] = lgb_model.predict(X_val)
    bal_acc_lgb = balanced_accuracy_score(y_val, oof_lgb[val_idx])
    print(f'    LGBM: {bal_acc_lgb:.6f}')
    
    test_xgb += xgb_model.predict_proba(X_test) / N_SPLITS
    test_lgb += lgb_model.predict_proba(X_test) / N_SPLITS

print(f'\n  OOF Balanced Accuracies:')
print(f'    XGB:  {balanced_accuracy_score(y, oof_xgb):.6f}')
print(f'    LGBM: {balanced_accuracy_score(y, oof_lgb):.6f}')

In [ ]:
print('\n[5/5] Creating submission...')

# Ensemble
test_ensemble = (test_xgb + test_lgb) / 2
test_pred = np.argmax(test_ensemble, axis=1)
test_labels = target_encoder.inverse_transform(test_pred)

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': test_labels
})

print(f'  Predictions: {submission["Irrigation_Need"].value_counts().to_dict()}')

submission.to_csv('submission.csv', index=False)
print('\n  ✓ submission.csv saved!')
print('\n' + '='*70)
print('🔥 DONE! Should score ~0.97-0.98+')
print('='*70)